In [1]:
import pandas as pd
import numpy as np

data = pd.read_parquet("../Dataset/Clean/Dataset_with_clusters.parquet")
sample = data.sample(10)
data.shape

(9800, 53)

## Entity & Keyword Extraction
This is an important component for textual analysis.
This Component has two key roles:
1. **High-Fidelity Information Extraction** [Involvement of NER]
2. **Semantic Tagging** [About NER and Semantic]

Standard Spacy models are often too general for news, <br>
which frequently contains niche organizations, emerging technologies, or specific political designations.

#### NER Model comparision for News Analysis
| Model        | Technique                         | Key Strength                                                                 | Performance (Speed)        |
|--------------|-----------------------------------|------------------------------------------------------------------------------|----------------------------|
| GLiNER       | Bi-Encoder / Zero-Shot            | Can extract any label you define on the fly without retraining.             | Medium (Better on GPU)     |
| Flair        | Character-Level Embeddings        | Industry standard for accuracy; handles typos and "news-speak" well.        | Slow (Sequential processing) |
| spaCy (TRF)  | Transformer-based (RoBERTa)       | Much stronger than the standard "sm" model; integrates with spaCy ecosystem | Medium-Fast                |
| SpanMarker   | PLM-based (BERT/RoBERTa)          | Current SOTA (State of the Art) for fixed-label NER accuracy.               | Medium                     |


In [2]:
query = data.query('word_count > 100').sample()['Content'].values[0]
print(query)

Bread has become the latest household to be slashed in cost in the supermarket price wars which have reduced milk to cheaper than bottled water. Branded loaves have now been reduced to as little as 75p and own-label bread can be bought for 55p in some stores, according to new research. Sainsbury's has led the battle by reducing the cost of a Hovis loaf to 75p - the lowest since prices of bread plummeted to similar levels in 2011. The price of loaves of bread has been slashed by supermarkets and has reached lows of 75p for branded loaves and 55p for supermarket own brands in the latest price wars, which last month cut the price of milk . The supermarket dropped 800g Hovis white and wholemeal loaves and 750g Best of Both from £1 and cut nine of its 800g own-label loaves from 75p to 55p. Trade magazine The Grocer said the price cuts come at a particularly bad time for bakers when the supply of wheat with the right levels of protein needed for baking have been tight and prices are expected

In [3]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp(query)
for ent in doc.ents:
    print(ent.text, ent.label_)
    
print("Length: ", len(doc.ents))

d:\Krishan\Project dataset\News categorization\.NewsEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Meghan Keneally PERSON
09:19 EST TIME
9 December 2012 DATE
08:51 EST TIME
10 December 2012 DATE
Hillary Clinton PERSON
Hamptons ORG
Bill PERSON
2016 DATE
State ORG
January DATE
New York GPE
Mrs Clinton PERSON
2016 DATE
Mrs Clinton PERSON
Republican NORP
the House ORG
Newt Gingrich PERSON
Hillary Clinton PERSON
State ORG
2016 DATE
2016 DATE
Hillary Clinton PERSON
Bill Clinton PERSON
Barack Obama PERSON
the Super Bowl EVENT
the Republican party ORG
today DATE
Meet The Press ORG
Gingrich PERSON
Bill Clinton PERSON
the 1990s DATE
Gingrich PERSON
Monica Lewinsky PERSON
First ORDINAL
Democrat NORP
Washington Post ORG
57 per cent MONEY
Americans NORP
Hillary Clinton PERSON
Mrs Clinton PERSON
Clintons PERSON
Clintons PERSON
the summer DATE
this past August DATE
Clinton PERSON
1999 DATE
Kate Capshaw PERSON
Stephen Spielberg PERSON
The New York Times ORG
one CARDINAL
first ORDINAL
Hamptons ORG
Bill PERSON
two CARDINAL
multi-million dollar MONEY
about 2016 CARDINAL
2008 DATE
September 30 of this 

In [4]:
nlp = spacy.load("en_core_web_trf")
doc = nlp(query)

print("List of Entities: \n")
for i,ent in enumerate(doc.ents):
    print(f"{i+1})",ent.text, "| Label: ",ent.label_, "-> ",spacy.explain(ent.label_))
    
print("Length: ", len(doc.ents))

spacy.displacy.render(doc, style="ent")

List of Entities: 

1) Meghan Keneally | Label:  PERSON ->  People, including fictional
2) 09:19 EST | Label:  TIME ->  Times smaller than a day
3) 2012 | Label:  DATE ->  Absolute or relative dates or periods
4) 08:51 EST | Label:  TIME ->  Times smaller than a day
5) 2012 | Label:  DATE ->  Absolute or relative dates or periods
6) Hillary Clinton | Label:  PERSON ->  People, including fictional
7) Hamptons | Label:  GPE ->  Countries, cities, states
8) Bill | Label:  PERSON ->  People, including fictional
9) 2016 | Label:  DATE ->  Absolute or relative dates or periods
10) State | Label:  ORG ->  Companies, agencies, institutions, etc.
11) January | Label:  DATE ->  Absolute or relative dates or periods
12) New York | Label:  GPE ->  Countries, cities, states
13) Clinton | Label:  PERSON ->  People, including fictional
14) 2016 | Label:  DATE ->  Absolute or relative dates or periods
15) Clinton | Label:  PERSON ->  People, including fictional
16) Republican | Label:  NORP ->  Nation

Transformer model of Spacy is little slow to load but it is a good model for NER.

Now, we will try other pre-trained Model which are different from spacy.

#### 1. `SpanMarker` <br>
SpanMarker is a framework for training powerful Named Entity Recognition models using familiar encoders such as BERT, RoBERTa and DeBERTa.

In [ ]:
# nlp = spacy.load("en_core_web_sm", exclude=["ner"])
# nlp.add_pipe("span_marker", config={"model": "tomaarsen/span-marker-bert-base-fewnerd-fine-super"})

# doc = nlp(query)
# print([(entity, entity.label_) for entity in doc.ents])

> Currently, Spanmarker is not compatable wiht current version of transformer. 

#### 2. **`Bert-base NER`**
distilbert-NER is the fine-tuned version of DistilBERT, which is a distilled variant of the BERT model. <br>
DistilBERT has fewer parameters than BERT, making it smaller, faster, and more efficient. distilbert-NER is specifically fine-tuned for the task of Named Entity Recognition (NER).

In [9]:
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

tokenizer = AutoTokenizer.from_pretrained("dslim/distilbert-NER")
model = AutoModelForTokenClassification.from_pretrained("dslim/distilbert-NER")

ner = pipeline("ner", model=model, tokenizer=tokenizer)
res = pd.DataFrame(ner(query))
res[res["entity"].str.contains("B")]

Device set to use cpu


,entity,score,index,word,start,end
0,B-PER,0.998210,3,Meg,5,8
1,B-PER,0.997226,4,##han,8,11
5,B-PER,0.975232,45,Hillary,110,117
7,B-LOC,0.661092,56,Hampton,160,167
9,B-PER,0.996857,61,Bill,187,191
10,B-PER,0.503599,74,Presidency,245,255
11,B-LOC,0.992094,106,New,405,408
15,B-MISC,0.973417,142,Republican,596,606
16,B-ORG,0.873213,146,House,622,627
17,B-PER,0.908750,147,New,628,631


This model is not performing well for our task. This does token level NER where it is missing many entities.

#### 3. **`Flair`**
Flair NER model to recognize 4 or 18 predefined entities.
| tag           |	meaning                 |
| ------------- | ------------------------- |
| CARDINAL      |	cardinal value          |
| DATE          |	date value              |
| EVENT         |	event name              |
| FAC           |	building name           |
| GPE           |	geo-political entity    |
| LANGUAGE      |	language name           |
| LAW           |	law name                |
| LOC           |	location name           |
| MONEY         |	money name              |
| NORP          |	affiliation             |
| ORDINAL       |	ordinal value           |
| ORG           |	organization name       |
| PERCENT       |	percent value           |
| PERSON        |	person name             |
| PRODUCT       |	product name            |
| QUANTITY      |	quantity value          |
| TIME          |	time value              |
| WORK_OF_ART   |	name of work of art     |

In [18]:
from flair.models import SequenceTagger
from flair.data import Sentence

tagger = SequenceTagger.load("flair/ner-english-ontonotes-fast")

2026-02-14 23:58:08,926 SequenceTagger predicts: Dictionary with 75 tags: O, S-PERSON, B-PERSON, E-PERSON, I-PERSON, S-GPE, B-GPE, E-GPE, I-GPE, S-ORG, B-ORG, E-ORG, I-ORG, S-DATE, B-DATE, E-DATE, I-DATE, S-CARDINAL, B-CARDINAL, E-CARDINAL, I-CARDINAL, S-NORP, B-NORP, E-NORP, I-NORP, S-MONEY, B-MONEY, E-MONEY, I-MONEY, S-PERCENT, B-PERCENT, E-PERCENT, I-PERCENT, S-ORDINAL, B-ORDINAL, E-ORDINAL, I-ORDINAL, S-LOC, B-LOC, E-LOC, I-LOC, S-TIME, B-TIME, E-TIME, I-TIME, S-WORK_OF_ART, B-WORK_OF_ART, E-WORK_OF_ART, I-WORK_OF_ART, S-FAC


In [19]:
sent = Sentence(query)

tagger.predict(sent)
# print(sent)

for entity in sent.get_spans("ner"):
    print(entity)

Span[2:4]: "Meghan Keneally" → PERSON (0.7062)
Span[8:10]: "09:19 EST" → TIME (0.6074)
Span[11:14]: "9 December 2012" → DATE (0.7100)
Span[20:22]: "08:51 EST" → TIME (0.6242)
Span[23:26]: "10 December 2012" → DATE (0.8122)
Span[27:29]: "Hillary Clinton" → PERSON (0.9809)
Span[38:39]: "Hamptons" → GPE (0.6951)
Span[42:43]: "Bill" → PERSON (0.9989)
Span[55:56]: "2016" → DATE (0.9969)
Span[64:65]: "State" → ORG (1.0000)
Span[71:72]: "January" → DATE (0.9963)
Span[83:85]: "New York" → GPE (0.9362)
Span[93:94]: "Clinton" → PERSON (0.9998)
Span[108:109]: "2016" → DATE (0.9990)
Span[111:112]: "Clinton" → PERSON (0.9994)
Span[118:119]: "Republican" → NORP (0.9985)
Span[122:123]: "House" → ORG (0.9979)
Span[123:125]: "Newt Gingrich" → PERSON (0.7935)
Span[151:153]: "Hillary Clinton" → PERSON (0.9460)
Span[171:172]: "State" → ORG (1.0000)
Span[185:186]: "2016" → DATE (0.9985)
Span[192:193]: "2016" → DATE (0.9979)
Span[194:196]: "Hillary Clinton" → PERSON (0.9655)
Span[199:201]: "Bill Clinton" → 

Flair NER is very good but it has high latency which is a problem for real-time application. <br>
High Efficiency but low throughput -> Can be used for smaller article where we need predefined class entities.

#### 4. **`GLiner`**
GLiNER(General Linguistic Named Entity Recognition) is a framework for training and deploying small Named Entity Recognition (NER) models with zero-shot capabilities. <br>
In addition to tradition NER, it also supports joint entity and relation extraction.

GLiner does not have a fixed "output head" for specific labels, it uses a **Bi-Encoder/Span-matching** archtecture.

In [ ]:
from gliner import GLiNER
from gliner2 import GLiNER2

ner_labels = [
    "Person", "Celebrity", "Political party", "Politician", "Activist", "Criminal", "Victim", "Witness",
    "Profession", "Job title", "Author", "Scientist", "Journalist", "Speaker", "Writer", "Artist",
    "Affiliation", "Organisation", "Company", "Startup", "Institution", "College", "University", "Government agency", "Military organisation", "Union", "Sports team", "Media",
    "Country", "City", "State", "Region", "Continent", "Climate zone", "Forest", "Desert", "Mountain", "Park", "Water body",
    "Building", "Airport", "Monument", "Landmark", "Date", "Time", "Duration", "Percent", "Money", "Temperature", "Speed", "Age",
    "Law", "Case", "Judge", "Constitution", "Election", 
    "Medical", "Disease", "Drug", "Symptom", "Chemical"
    "Location", "Product", "Event", "Work_of_art", "Language",
    "Business", "Market", "Stock", "Currency", "Product"
    "Sport", "Games", "Award", "Event",
    "Art", "Book", "Movie", "TV show", 
    "Computer", "Vehicle", "Machine", "Programming language", "Technology",
    "Color", "Shape", "Size", "Weight", "Weapon", "Battle", "Natural disaster",
    "Quantity", "Ordinal", "Cardinal", 
    "Animal", "Plant", "Organism",
]

In [14]:
# Loading model
gliner_model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")

entities = gliner_model.predict_entities(query, labels=ner_labels, threshold=0.5)
# Display predicted entities and their labels
for entity in entities:
    print(entity["text"], "=>", entity["label"])

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 74367.09it/s]
d:\Krishan\Project dataset\News categorization\.NewsEnv\Lib\site-packages\transformers\convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
d:\Krishan\Project dataset\News categorization\.NewsEnv\Lib\site-packages\gliner\data_processing\processor.py:395: UserWarning: Sentence of length 468 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to

Bread => Product
price wars => Battle
milk => Product
Branded loaves => Product
75p => Quantity
own-label bread => Product
55p => Quantity
Sainsbury's => Organisation
Hovis loaf => Product
75p => Quantity
bread => Product
2011 => Date
bread => Product
supermarkets => Organisation
75p => Currency
branded loaves => Product
55p => Quantity
price wars => Battle
milk => Product
800g => Quantity
Hovis => Product
white and wholemeal loaves => Product
750g => Quantity
Best of Both => Product
£1 => Currency
800g => Quantity
own-label loaves => Product
75p => Currency
55p => Quantity
The Grocer => Market
price cuts => Event
bakers => Profession
wheat => Product
August => Date
4.5 per cent => Percent
bakers => Profession
milling industry insider => Profession
bakers => Profession
BrandView.com => Organisation
bread => Product
Sainsbury's => Organisation
big four supermarket chains => Organisation
Hovis => Product
Waitrose => Organisation
£1 => Currency
mid-January => Time
The Grocer => Market
bra

In [16]:
# Loading model
gliner_model = GLiNER2.from_pretrained("fastino/gliner2-large-v1")
entities = gliner_model.extract_entities(query, entity_types=ner_labels)

for label, ent in entities["entities"].items():
    if ent:
        print(f"{label}: {ent}")

🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-large
Counting layer     : count_lstm
Token pooling      : first
Profession: ['dairy farmers', 'bakers']
Company: ['Asda', "Sainsbury's", "Morrison's", 'supermarkets', 'Waitrose', 'Kingsmill', 'big four supermarket chains', 'BrandView.com']
Date: ['2011', 'August', 'mid-January', 'last month']
Percent: ['4.5 per cent']
Money: ['base price']
Product: ['bread', 'milk', 'Hovis loaf', 'bottled water']
Event: ['supermarket price wars', 'price wars']
Business: ['business', 'livelihoods']
Market: ['stores']
Currency: ['£1']
Color: ['white', 'wholemeal']
Weight: ['800g']
Battle: ['battle']


Gliner is the best model for Recoginition of Named Entities with custom class label. <br>
It understand the semantic meaning of searched article and each label and finds matching entities.

Now, we will extract keywords from our given article. <br>

For keyword extraction, there is one clear cut winner -> 
### ***`KeyBERT`***
KeyBERT is fast library for keyword and keyphrase extraction that utilizes BERT embeddings to find sub-phrases most similar to a document. <br>
It calculates cosine similarity between document and n-gram embeddings to identify the most relevant terms.

##### <b> Key Features </b>
- Methodology: Uses BERT to generate document embeddings, extract N-gram token & compute cosine similarity to rank them.
- Customization: Allows tuning of top-n keywords, N-gram range & diversity algrorithm. [Maximal Marginal Relevance]
- Flexibility: Support various backends(PyTorch, Tensorflow, JAX, Huggign Face) & allows for multilingual models
- Integration: Easily intergrate with other Bert based specialised models
- Performance: Fast & memory efficient

In [17]:
from keybert import KeyBERT
keybert = KeyBERT(model = 'all-MiniLM-L6-v2')
keywords = keybert.extract_keywords(query, keyphrase_ngram_range=(1, 2), 
    stop_words="english",
    use_mmr=True, # Maximal Marginal Relevance - for diversity
    diversity=0.5, # 0.0 to 1.0
    top_n=10, # Number of keywords
)
keywords

[('bread price', 0.7214),
 ('label loaves', 0.5386),
 ('cost hovis', 0.4764),
 ('milk cheaper', 0.4573),
 ('75p sainsbury', 0.4542),
 ('price cuts', 0.4519),
 ('supply wheat', 0.4268),
 ('supermarket dropped', 0.3785),
 ('farmers criticised', 0.2531),
 ('reducing', 0.2116)]

This is the best model for NER+keyword extraction. <br>
We will optimize this model for our project inference pipeline.

Now, There are a lot of various pipelines and libraries out there for NER+keyword extraction but we dont know which one if correct for our task.

Also, NER model is not something we can fine-tune as it is blind implementation.
So, we need to foucs on evaluation for NER.

But, There is no direct correct way for evaluation for NER task. <br>
So, a combination of various method/trick/metrics will be used in combination.

Evaluation Pipeline:
Design a class for NER Evaluation.
1. Intrinsic proxy metrics: Precision, Recall, F1 vs summary-entities
2. Per-class stats: entity distribution vs topic clusters
3. Efficiency: Latency, Throughput
4. Extrinsic metrics: topic coherence delta, clustering purity, keyword precision@k
5. Final combined score

In [13]:
import time
import numpy as np
from collections import defaultdict, Counter
from sklearn.metrics import normalized_mutual_info_score
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import entropy

In [14]:
class NEREvaluator:
    def __init__(self, model, model_type="spacy"):
        """
        model_type: "spacy" | "HuggingFace" | "flair" | "gliner"
        model: loaded NER model
        """
        self.model = model
        self.model_type = model_type.lower()
        
    # 1. Internal NER call
    def _extract_entities(self, text):

        if self.model_type == "spacy":
            doc = self.model(text)
            return [
                {"start": ent.start_char, "end": ent.end_char, "label": ent.label_, "text": ent.text}
                for ent in doc.ents
            ]

        elif self.model_type == "huggingface":
            outputs = self.model(text)
            return [
                {"start": ent["start"], "end": ent["end"], "label": ent["entity"], "text": ent["word"]}
                for ent in outputs
            ]

        elif self.model_type == "flair":
            from flair.data import Sentence
            sent = Sentence(text)
            self.model.predict(sent)
            return [
                {"start": ent.start_position, "end": ent.end_position, "label": ent.tag, "text": ent.text}
                for ent in sent.get_spans("ner")
            ]

        elif self.model_type == "gliner":
            ents = self.model.predict_entities(text)
            return [
                {"start": ent["start"], "end": ent["end"], "label": ent["label"], "text": ent["text"]}
                for ent in ents
            ]

        else:
            raise ValueError("Unsupported model_type")
        
    # 2. Run model
    def run_model(self, df, text_col="Content"):
        preds = []
        start = time.time()

        for text in df[text_col].astype(str):
            ents = self._extract_entities(text)
            preds.append(ents)

        end = time.time()

        total_time = end - start
        latency = total_time / max(1, len(df))
        throughput = len(df) / max(1e-9, total_time)

        return preds, latency, throughput
    
    # 3. Entity Density
    def entity_density(self, preds, dataset):
        densities = []
        for ents, wc in zip(preds, dataset["word_count"]):
            densities.append(len(ents) / max(1, wc))
        return np.mean(densities)
            
    # 4. Entity Diversity
    def entity_diversity(self, preds):
        all_ents = [e["text"].lower() for doc in preds for e in doc]
        if len(all_ents) == 0:
            return 0.0
        return len(set(all_ents)) / len(all_ents)
    
    # 5. Label Entropy
    def label_entropy(self, preds):
        labels = [e["label"] for doc in preds for e in doc]
        if len(labels) == 0:
            return 0.0
        counts = np.array(list(Counter(labels).values()))
        probs = counts / counts.sum()
        return entropy(probs)
    
    # 6. Final score
    def final_score(self, m, weights=None):
        weights = {
            "density": 0.3,
            "diversity": 0.25,
            "entropy": 0.2,
            "latency" : -0.15,
            "throughput": 0.1,
        }
        
        score = 0
        for k,w in weights.items():
            score += m[k]*w
        
        return score
    
    # Main Evaluation
    def evaluate(self, df, text_col="Content"):
        preds, latency, throughput = self.run_model(df, text_col)

        metrics = {
            "density": self.entity_density(preds, df),
            "diversity": self.entity_diversity(preds),
            "entropy": self.label_entropy(preds),
            "latency": latency,
            "throughput": throughput,
        }

        metrics["score"] = self.final_score(metrics)
        
        return metrics
    

In [27]:
nlp = spacy.load("en_core_web_sm")
sample = data.sample(1000)

evaluator = NEREvaluator(model=nlp, model_type="spacy")
metrics = evaluator.evaluate(sample)

for k,v in metrics.items():
    print(f"{k:15s}: {round(v, 4)}")

density        : 0.1124
diversity      : 0.3437
entropy        : 2.1273
latency        : 0.0554
throughput     : 18.0394
score          : 2.3407


##### Approx evaluation for spacy small model:

|   Metrics      |  Value  |
|--------------- |---------|
| density        | 0.1124  |
| diversity      | 0.3437  |
| entropy        | 2.1273  |
| latency        | 0.0554  |
| throughput     | 18.0394 |
| score          | 2.3407  |

We have already designed our semantic search and clustering pipeline but <br>
maybe we can improve them using NER in the future.

In [17]:
eval = NEREvaluator(model=tagger, model_type="flair")
metrics = eval.evaluate(sample)

for k,v in metrics.items():
    print(f"{k:15s}: {round(v, 4)}")

density        : 0.108
diversity      : 0.5924
entropy        : 2.0601
latency        : 0.4213
throughput     : 2.3735
score          : 0.7667
